In [1]:
!pip install transformers torch scikit-learn -q

print("Libraries installed")

Libraries installed


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import torch
import os
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    RobertaTokenizer, RobertaForSequenceClassification,
    Trainer, TrainingArguments
)
from torch.utils.data import Dataset
import warnings
warnings.filterwarnings('ignore')

# ── Check GPU ──────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — training will be very slow")
    print("Go to Runtime → Change runtime type → GPU")

# ── Paths ──────────────────────────────────────────
LABELLED_PATH = "/content/drive/MyDrive/Original Reddit Data/Labelled Data"
SAVE_PATH     = "/content/drive/MyDrive/"
OUTPUT_PATH   = "/content/drive/MyDrive/figures/"
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("\nSetup complete")

Mounted at /content/drive
Device: cuda
GPU: Tesla T4

Setup complete


In [3]:
# ── Load Part B ────────────────────────────────────
import re

def clean_for_bert(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower().strip()
    return text

label_files = {
    "Drug and Alcohol" : LABELLED_PATH + "/LD DA 1.csv",
    "Early Life"       : LABELLED_PATH + "/LD EL1.csv",
    "Personality"      : LABELLED_PATH + "/LD PF1.csv",
    "Trauma and Stress": LABELLED_PATH + "/LD TS 1.csv"
}

dfs_b = []
for label, filepath in label_files.items():
    try:
        temp = pd.read_csv(filepath)
        temp['Label'] = label
        dfs_b.append(temp)
        print(f"✓ {label}: {temp.shape[0]} rows")
    except Exception as e:
        print(f"✗ {label}: {e}")

df_b = pd.concat(dfs_b, ignore_index=True)
df_b = df_b.dropna(subset=['selftext'])
if 'CAT 1' in df_b.columns:
    df_b = df_b.drop(columns=['CAT 1'])

df_b['full_text'] = (
    df_b['title'].fillna('') + ' ' +
    df_b['selftext'].fillna('')
)
df_b['bert_text'] = df_b['full_text'].apply(clean_for_bert)

# ── Create label mapping ───────────────────────────
label2id = {
    'Drug and Alcohol' : 0,
    'Early Life'       : 1,
    'Personality'      : 2,
    'Trauma and Stress': 3
}
id2label = {v: k for k, v in label2id.items()}

df_b['label_id'] = df_b['Label'].map(label2id)

print(f"\nTotal posts: {df_b.shape[0]}")
print(f"Label distribution:")
print(df_b['Label'].value_counts())
print(f"\nLabel mapping: {label2id}")

✓ Drug and Alcohol: 223 rows
✓ Early Life: 200 rows
✓ Personality: 200 rows
✓ Trauma and Stress: 200 rows

Total posts: 800
Label distribution:
Label
Drug and Alcohol     200
Early Life           200
Personality          200
Trauma and Stress    200
Name: count, dtype: int64

Label mapping: {'Drug and Alcohol': 0, 'Early Life': 1, 'Personality': 2, 'Trauma and Stress': 3}


In [4]:
# ── Stratified 80/20 split ─────────────────────────
train_df, test_df = train_test_split(
    df_b,
    test_size=0.2,
    random_state=42,
    stratify=df_b['label_id']
)

print(f"Train set: {len(train_df)} posts")
print(f"Test set:  {len(test_df)} posts")
print(f"\nTrain label distribution:")
print(train_df['Label'].value_counts())
print(f"\nTest label distribution:")
print(test_df['Label'].value_counts())

Train set: 640 posts
Test set:  160 posts

Train label distribution:
Label
Drug and Alcohol     160
Early Life           160
Personality          160
Trauma and Stress    160
Name: count, dtype: int64

Test label distribution:
Label
Personality          40
Early Life           40
Trauma and Stress    40
Drug and Alcohol     40
Name: count, dtype: int64


In [5]:
# ── PyTorch Dataset ────────────────────────────────
class MentalHealthDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: val[idx]
            for key, val in self.encodings.items()
        }
        item['labels'] = self.labels[idx]
        return item

print("Dataset class defined")

Dataset class defined


In [6]:
# ── Evaluation metrics ─────────────────────────────
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro'
    )
    acc = accuracy_score(labels, preds)

    return {
        'accuracy' : round(acc, 4),
        'f1'       : round(f1, 4),
        'precision': round(precision, 4),
        'recall'   : round(recall, 4)
    }

print("Metrics function defined")

Metrics function defined


In [14]:
# ── Fine-tune BERT ─────────────────────────────────
print("Loading BERT tokenizer and model...")

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=4,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# Create datasets
bert_train = MentalHealthDataset(
    train_df['bert_text'], train_df['label_id'], bert_tokenizer
)
bert_test = MentalHealthDataset(
    test_df['bert_text'], test_df['label_id'], bert_tokenizer
)

# Training arguments
bert_args = TrainingArguments(
    output_dir                  = '/content/bert_results',
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    logging_steps               = 10,
    report_to                   = 'none'
)

# Trainer
bert_trainer = Trainer(
    model           = bert_model,
    args            = bert_args,
    train_dataset   = bert_train,
    eval_dataset    = bert_test,
    compute_metrics = compute_metrics
)

print("Training BERT...")
print("Expected time: 20-25 minutes on GPU\n")
bert_trainer.train()
print("\nBERT training complete ✅")

Loading BERT tokenizer and model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training BERT...
Expected time: 20-25 minutes on GPU



Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.373076,1.294581,0.381200,0.366600,0.373900,0.381200
2,1.078924,1.100568,0.556300,0.526500,0.555100,0.556200
3,0.899329,1.004175,0.637500,0.628800,0.635400,0.637500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


BERT training complete ✅


In [15]:
# ── Evaluate BERT ──────────────────────────────────
print("Evaluating BERT on test set...")

bert_preds = bert_trainer.predict(bert_test)
bert_pred_labels = bert_preds.predictions.argmax(-1)
bert_true_labels = bert_preds.label_ids

bert_report = classification_report(
    bert_true_labels,
    bert_pred_labels,
    target_names=list(label2id.keys()),
    output_dict=True
)

print("BERT Classification Report:")
print(classification_report(
    bert_true_labels,
    bert_pred_labels,
    target_names=list(label2id.keys())
))

# Save results
bert_results = {
    'model'    : 'BERT',
    'accuracy' : round(accuracy_score(bert_true_labels, bert_pred_labels), 4),
    'f1_macro' : round(bert_report['macro avg']['f1-score'], 4),
    'precision': round(bert_report['macro avg']['precision'], 4),
    'recall'   : round(bert_report['macro avg']['recall'], 4),
    'report'   : bert_report
}

print(f"\nBERT Summary:")
print(f"  Accuracy:  {bert_results['accuracy']}")
print(f"  F1 Macro:  {bert_results['f1_macro']}")
print(f"  Precision: {bert_results['precision']}")
print(f"  Recall:    {bert_results['recall']}")

Evaluating BERT on test set...


BERT Classification Report:
                   precision    recall  f1-score   support

 Drug and Alcohol       0.71      0.75      0.73        40
       Early Life       0.67      0.80      0.73        40
      Personality       0.62      0.40      0.48        40
Trauma and Stress       0.55      0.60      0.57        40

         accuracy                           0.64       160
        macro avg       0.64      0.64      0.63       160
     weighted avg       0.64      0.64      0.63       160


BERT Summary:
  Accuracy:  0.6375
  F1 Macro:  0.6288
  Precision: 0.6354
  Recall:    0.6375


In [17]:
# ── Fine-tune RoBERTa ──────────────────────────────
import gc
gc.collect()
torch.cuda.empty_cache()

print("Loading RoBERTa tokenizer and model...")

roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
roberta_model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=4,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

roberta_train = MentalHealthDataset(
    train_df['bert_text'], train_df['label_id'], roberta_tokenizer
)
roberta_test = MentalHealthDataset(
    test_df['bert_text'], test_df['label_id'], roberta_tokenizer
)

roberta_args = TrainingArguments(
    output_dir                  = '/content/roberta_results',
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    logging_steps               = 10,
    report_to                   = 'none'
)

roberta_trainer = Trainer(
    model           = roberta_model,
    args            = roberta_args,
    train_dataset   = roberta_train,
    eval_dataset    = roberta_test,
    compute_metrics = compute_metrics
)

print("Training RoBERTa...")
print("Expected time: 20-25 minutes on GPU\n")
roberta_trainer.train()
print("\nRoBERTa training complete ✅")

Loading RoBERTa tokenizer and model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training RoBERTa...
Expected time: 20-25 minutes on GPU



Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.076756,1.005373,0.556300,0.528200,0.643600,0.556200
2,0.607707,0.750219,0.718800,0.707900,0.735600,0.718800
3,0.460767,0.681990,0.756200,0.751000,0.752800,0.756200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


RoBERTa training complete ✅


In [18]:
# ── Evaluate RoBERTa ───────────────────────────────
print("Evaluating RoBERTa on test set...")

roberta_preds = roberta_trainer.predict(roberta_test)
roberta_pred_labels = roberta_preds.predictions.argmax(-1)
roberta_true_labels = roberta_preds.label_ids

roberta_report = classification_report(
    roberta_true_labels,
    roberta_pred_labels,
    target_names=list(label2id.keys()),
    output_dict=True
)

print("RoBERTa Classification Report:")
print(classification_report(
    roberta_true_labels,
    roberta_pred_labels,
    target_names=list(label2id.keys())
))

roberta_results = {
    'model'    : 'RoBERTa',
    'accuracy' : round(accuracy_score(roberta_true_labels, roberta_pred_labels), 4),
    'f1_macro' : round(roberta_report['macro avg']['f1-score'], 4),
    'precision': round(roberta_report['macro avg']['precision'], 4),
    'recall'   : round(roberta_report['macro avg']['recall'], 4),
    'report'   : roberta_report
}

print(f"\nRoBERTa Summary:")
print(f"  Accuracy:  {roberta_results['accuracy']}")
print(f"  F1 Macro:  {roberta_results['f1_macro']}")
print(f"  Precision: {roberta_results['precision']}")
print(f"  Recall:    {roberta_results['recall']}")

Evaluating RoBERTa on test set...


RoBERTa Classification Report:
                   precision    recall  f1-score   support

 Drug and Alcohol       0.79      0.93      0.85        40
       Early Life       0.79      0.85      0.82        40
      Personality       0.76      0.62      0.68        40
Trauma and Stress       0.68      0.62      0.65        40

         accuracy                           0.76       160
        macro avg       0.75      0.76      0.75       160
     weighted avg       0.75      0.76      0.75       160


RoBERTa Summary:
  Accuracy:  0.7562
  F1 Macro:  0.751
  Precision: 0.7528
  Recall:    0.7562


In [19]:
# ── Save all results ───────────────────────────────
import json

# Save BERT predictions
bert_pred_df = test_df.copy()
bert_pred_df['bert_predicted'] = [id2label[p] for p in bert_pred_labels]
bert_pred_df['bert_correct']   = bert_pred_df['Label'] == bert_pred_df['bert_predicted']
bert_pred_df.to_csv(SAVE_PATH + "bert_predictions.csv", index=False)
print("✓ bert_predictions.csv saved")

# Save RoBERTa predictions
roberta_pred_df = test_df.copy()
roberta_pred_df['roberta_predicted'] = [id2label[p] for p in roberta_pred_labels]
roberta_pred_df['roberta_correct']   = roberta_pred_df['Label'] == roberta_pred_df['roberta_predicted']
roberta_pred_df.to_csv(SAVE_PATH + "roberta_predictions.csv", index=False)
print("✓ roberta_predictions.csv saved")

# Save metrics summary
metrics_summary = {
    'BERT': {
        'accuracy' : 0.6375,
        'f1_macro' : 0.6288,
        'precision': 0.6354,
        'recall'   : 0.6375
    },
    'RoBERTa': {
        'accuracy' : 0.7562,
        'f1_macro' : 0.7510,
        'precision': 0.7528,
        'recall'   : 0.7562
    }
}

with open(SAVE_PATH + "model_metrics.json", 'w') as f:
    json.dump(metrics_summary, f, indent=2)
print("✓ model_metrics.json saved")

print("\n=== FINAL COMPARISON ===")
print(f"{'Model':<12} {'Accuracy':>10} {'F1 Macro':>10} {'Precision':>10} {'Recall':>10}")
print("-" * 55)
print(f"{'BERT':<12} {'0.6375':>10} {'0.6288':>10} {'0.6354':>10} {'0.6375':>10}")
print(f"{'RoBERTa':<12} {'0.7562':>10} {'0.7510':>10} {'0.7528':>10} {'0.7562':>10}")
print("\n✓ All results saved")
print("Notebook 04 Complete ✅")

✓ bert_predictions.csv saved
✓ roberta_predictions.csv saved
✓ model_metrics.json saved

=== FINAL COMPARISON ===
Model          Accuracy   F1 Macro  Precision     Recall
-------------------------------------------------------
BERT             0.6375     0.6288     0.6354     0.6375
RoBERTa          0.7562     0.7510     0.7528     0.7562

✓ All results saved
Notebook 04 Complete ✅
